In [60]:
import pandas as pd
import sqlite3
# import requests
# import zipfile
# import io
from google.cloud import bigquery
# data from 1992 to 2018

In [2]:
# # url for zipped data
# url = 'https://www.fs.usda.gov/rds/archive/products/RDS-2013-0009.6/RDS-2013-0009.6_SQLITE.zip'
# # get zipped data
# response = requests.get(url, stream=True)
# z = zipfile.ZipFile(io.BytesIO(response.content))

In [3]:
# # extract necessary files
# z.extract('Data/FPA_FOD_20221014.sqlite')
# z.extract('Data/_variable_descriptions.csv')

In [61]:
# original pandas_gbq query
long_sql_aq = """
SELECT poc,latitude,longitude,parameter_name,sample_duration,
date_local,units_of_measure,observation_count,
arithmetic_mean,county_code,site_num
FROM `bigquery-public-data.epa_historical_air_quality.co_daily_summary`
WHERE state_code = '06' and date_local < '2019-01-01' and date_local > '1996-12-31' and datum != 'UNKNOWN' and datum != 'NAD27'
UNION ALL
SELECT poc,latitude,longitude,parameter_name,sample_duration,
date_local,units_of_measure,observation_count,
arithmetic_mean,county_code,site_num
FROM `bigquery-public-data.epa_historical_air_quality.pressure_daily_summary`
WHERE state_code = '06' and date_local < '2019-01-01' and date_local > '1996-12-31' and datum != 'UNKNOWN' and datum != 'NAD27'
UNION ALL
SELECT poc,latitude,longitude,parameter_name,sample_duration,
date_local,units_of_measure,observation_count,
arithmetic_mean,county_code,site_num
FROM `bigquery-public-data.epa_historical_air_quality.rh_and_dp_daily_summary`
WHERE state_code = '06' and date_local < '2019-01-01' and date_local > '1996-12-31' and datum != 'UNKNOWN' and datum != 'NAD27'
UNION ALL
SELECT poc,latitude,longitude,parameter_name,sample_duration,
date_local,units_of_measure,observation_count,
arithmetic_mean,county_code,site_num
FROM `bigquery-public-data.epa_historical_air_quality.temperature_daily_summary`
WHERE state_code = '06' and date_local < '2019-01-01' and date_local > '1996-12-31' and datum != 'UNKNOWN' and datum != 'NAD27'
UNION ALL
SELECT poc,latitude,longitude,parameter_name,sample_duration,
date_local,units_of_measure,observation_count,
arithmetic_mean,county_code,site_num
FROM `bigquery-public-data.epa_historical_air_quality.wind_daily_summary`
WHERE state_code = '06' and date_local < '2019-01-01' and date_local > '1996-12-31' and datum != 'UNKNOWN' and datum != 'NAD27'
"""

long_sql_pt = """
SELECT  
    measurement_year,
    tree_county_code,latitude,longitude,elevation,
    trees_per_acre_unadjusted,
    water_on_plot_code_name,
    species_common_name,species_group_code_name,
    total_height,
    current_diameter,
    tree_status_code_name,
    invasive_sampling_status_code_name,
FROM `bigquery-public-data.usfs_fia.plot_tree`
WHERE plot_state_code = 6 AND total_height > 0 AND measurement_year > 1996
"""


In [ ]:
# test_aq = bigquery.Client().query(long_sql_aq,project='bigquery-public-data').to_dataframe()
# test_aq

In [5]:
sql_aq = """
SELECT *
FROM `california_air_trees_fires.epa_aq_ca5`
"""

sql_pt = """
SELECT *
FROM `california_air_trees_fires.usfs_fia_pt_ca2`
"""

## Air Quality

In [6]:
# pandas gbq will auth with google account first, rerun after
aq = pd.read_gbq(sql_aq, dialect='standard', project_id='my-ds-projects', use_bqstorage_api=True)

In [7]:
aq.date_local = aq.date_local.astype('datetime64[ns]')
aq

,poc,latitude,longitude,parameter_name,sample_duration,date_local,units_of_measure,observation_count,arithmetic_mean,county_code,site_num
0,1,39.202946,-122.018028,Wind Speed - Resultant,1 HOUR,1999-03-28,Knots,24,2.916667,011,0002
1,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-03-22,Knots,24,3.541667,011,1001
2,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-04-19,Knots,24,3.625000,011,1001
3,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-10-28,Knots,24,5.500000,011,1001
4,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-11-08,Knots,24,5.666667,011,1001
...,...,...,...,...,...,...,...,...,...,...,...
5150901,1,34.051110,-118.456360,Carbon monoxide,1 HOUR,2015-09-13,Parts per million,23,0.230435,037,0113
5150902,1,34.051110,-118.456360,Carbon monoxide,1 HOUR,2015-12-06,Parts per million,22,0.427273,037,0113
5150903,1,34.051110,-118.456360,Carbon monoxide,8-HR RUN AVG END HOUR,2015-03-23,Parts per million,24,0.275000,037,0113
5150904,1,34.051110,-118.456360,Barometric pressure,1 HOUR,2018-06-24,Millibars,24,1002.458333,037,0113


In [8]:
aq['year'] = aq.date_local.dt.year
aq.county_code = aq.county_code.astype(int)
aq

,poc,latitude,longitude,parameter_name,sample_duration,date_local,units_of_measure,observation_count,arithmetic_mean,county_code,site_num,year
0,1,39.202946,-122.018028,Wind Speed - Resultant,1 HOUR,1999-03-28,Knots,24,2.916667,11,0002,1999
1,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-03-22,Knots,24,3.541667,11,1001,1999
2,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-04-19,Knots,24,3.625000,11,1001,1999
3,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-10-28,Knots,24,5.500000,11,1001,1999
4,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-11-08,Knots,24,5.666667,11,1001,1999
...,...,...,...,...,...,...,...,...,...,...,...,...
5150901,1,34.051110,-118.456360,Carbon monoxide,1 HOUR,2015-09-13,Parts per million,23,0.230435,37,0113,2015
5150902,1,34.051110,-118.456360,Carbon monoxide,1 HOUR,2015-12-06,Parts per million,22,0.427273,37,0113,2015
5150903,1,34.051110,-118.456360,Carbon monoxide,8-HR RUN AVG END HOUR,2015-03-23,Parts per million,24,0.275000,37,0113,2015
5150904,1,34.051110,-118.456360,Barometric pressure,1 HOUR,2018-06-24,Millibars,24,1002.458333,37,0113,2018


In [9]:
aq.head()

,poc,latitude,longitude,parameter_name,sample_duration,date_local,units_of_measure,observation_count,arithmetic_mean,county_code,site_num,year
0,1,39.202946,-122.018028,Wind Speed - Resultant,1 HOUR,1999-03-28,Knots,24,2.916667,11,0002,1999
1,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-03-22,Knots,24,3.541667,11,1001,1999
2,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-04-19,Knots,24,3.625000,11,1001,1999
3,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-10-28,Knots,24,5.500000,11,1001,1999
4,1,39.012396,-122.092471,Wind Speed - Resultant,1 HOUR,1999-11-08,Knots,24,5.666667,11,1001,1999


In [10]:
aq.year.value_counts().sort_index().index.max(),aq.year.value_counts().sort_index().index.min()

(2018, 1997)

## Tree Plots

In [11]:
# pandas gbq will auth with google account first, rerun after
pt = pd.read_gbq(sql_pt, dialect='standard', project_id='my-ds-projects', use_bqstorage_api=True)

In [12]:
pt.head()

,measurement_year,tree_county_code,latitude,longitude,elevation,trees_per_acre_unadjusted,water_on_plot_code_name,species_common_name,species_group_code_name,total_height,current_diameter,tree_status_code_name,invasive_sampling_status_code_name
0,2013,51,37.586124,-118.186569,10500,0.999188,None - no water sources within the accessible ...,Great Basin bristlecone pine,Other western softwoods,39,40.299999,Live tree,Invasive plant data collected on all accessibl...
1,2011,51,37.733501,-118.257523,9900,0.999188,None - no water sources within the accessible ...,limber pine,Other western softwoods,43,26.400000,Live tree,Invasive plant data collected on all accessibl...
2,2006,107,36.351498,-118.465668,10700,0.999188,Permanent streams or ponds too small to qualif...,foxtail pine,Other western softwoods,64,37.299999,Live tree,Not collecting invasive plant data
3,2006,107,36.511063,-118.471970,10900,6.018046,None - no water sources within the accessible ...,foxtail pine,Other western softwoods,29,8.900000,Live tree,Not collecting invasive plant data
4,2007,71,34.125198,-116.906303,10100,0.999188,None - no water sources within the accessible ...,limber pine,Other western softwoods,113,45.000000,Dead tree,Invasive plant data collected on all accessibl...


In [13]:
pt = pt[pt['tree_county_code'].notna()].copy()
pt.columns

Index(['measurement_year', 'tree_county_code', 'latitude', 'longitude',
       'elevation', 'trees_per_acre_unadjusted', 'water_on_plot_code_name',
       'species_common_name', 'species_group_code_name', 'total_height',
       'current_diameter', 'tree_status_code_name',
       'invasive_sampling_status_code_name'],
      dtype='object')

In [14]:
pt.head()

,measurement_year,tree_county_code,latitude,longitude,elevation,trees_per_acre_unadjusted,water_on_plot_code_name,species_common_name,species_group_code_name,total_height,current_diameter,tree_status_code_name,invasive_sampling_status_code_name
0,2013,51,37.586124,-118.186569,10500,0.999188,None - no water sources within the accessible ...,Great Basin bristlecone pine,Other western softwoods,39,40.299999,Live tree,Invasive plant data collected on all accessibl...
1,2011,51,37.733501,-118.257523,9900,0.999188,None - no water sources within the accessible ...,limber pine,Other western softwoods,43,26.400000,Live tree,Invasive plant data collected on all accessibl...
2,2006,107,36.351498,-118.465668,10700,0.999188,Permanent streams or ponds too small to qualif...,foxtail pine,Other western softwoods,64,37.299999,Live tree,Not collecting invasive plant data
3,2006,107,36.511063,-118.471970,10900,6.018046,None - no water sources within the accessible ...,foxtail pine,Other western softwoods,29,8.900000,Live tree,Not collecting invasive plant data
4,2007,71,34.125198,-116.906303,10100,0.999188,None - no water sources within the accessible ...,limber pine,Other western softwoods,113,45.000000,Dead tree,Invasive plant data collected on all accessibl...


In [15]:
pt.measurement_year.value_counts().sort_index().index.max(),pt.measurement_year.value_counts().sort_index().index.min()

(2018, 2001)

In [16]:
# pt.pivot_table(columns='species_common_name',index=['measurement_year','tree_county_code'],
#         values=['latitude','longitude','elevation','trees_per_acre_unadjusted','water_on_plot_code_name',
#                 'species_group_code_name','total_height','current_diameter','tree_status_code_name','invasive_sampling_status_code_name']).reset_index()
# piv_pt = pt.pivot_table(columns='species_group_code_name',index=['measurement_year','tree_county_code'],
#         values=['latitude','longitude','elevation','trees_per_acre_unadjusted','water_on_plot_code_name',
#                 'total_height','current_diameter','tree_status_code_name','invasive_sampling_status_code_name']).reset_index()


In [17]:
# # california plots
# # pd.read_csv('https://apps.fs.usda.gov/fia/datamart/CSV/CA_PLOT.csv')
# cap = pd.read_csv('Data/CA_PLOT.csv')

# cap_col_idx = [x-1 for x in (
#     [1]+
#     list(range(12,15))+
#     list(range(18,23))+
#     [28,54]+
#     list(range(59,64))+
#     list(range(65,70))
#     )]

# # california trees
# # pd.read_csv('https://apps.fs.usda.gov/fia/datamart/CSV/CA_TREE.csv')
# cat = pd.read_csv('Data/CA_TREE.csv',dtype={'P2A_GRM_FLG':str,'GST_PNWRS':str})

# cat_col_idx = [x-1 for x in (
#     [2,4,7]+
#     list(range(15,19))+
#     [20,21,22,24,25]+
#     list(range(145,148))
#     )]


In [18]:
# cat.columns[cat_col_idx]

In [19]:
# cap = cap[cap.columns[cap_col_idx]]
# cat = cat[cat.columns[cat_col_idx]]
# # california trees n plots merged
# capt = pd.merge(cat,cap,'left',left_on='PLT_CN',right_on='CN').drop(columns=['PLT_CN','CN'])
# capt = capt[(capt.notnull().sum() > 0)[capt.notnull().sum() > 0].index]
# # capt

In [20]:
# capt.columns

## Fires

In [21]:
conn = sqlite3.connect('Data/FPA_FOD_20221014.sqlite')

In [22]:
fire = pd.read_sql('''
                    select 
                    *
                    from fires where STATE = :state
                    ''',conn,params={'state':'CA'})
fire = fire.drop(columns='Shape')
fire

,OBJECTID,FOD_ID,FPA_ID,SOURCE_SYSTEM_TYPE,SOURCE_SYSTEM,NWCG_REPORTING_AGENCY,NWCG_REPORTING_UNIT_ID,NWCG_REPORTING_UNIT_NAME,SOURCE_REPORTING_UNIT,SOURCE_REPORTING_UNIT_NAME,...,CONT_TIME,FIRE_SIZE,FIRE_SIZE_CLASS,LATITUDE,LONGITUDE,OWNER_DESCR,STATE,COUNTY,FIPS_CODE,FIPS_NAME
0,1,1,FS-1418826,FED,FS-FIRESTAT,FS,USCAPNF,Plumas National Forest,0511,Plumas National Forest,...,1730,0.10,A,40.036944,-121.005833,USFS,CA,63,06063,Plumas County
1,2,2,FS-1418827,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,...,1530,0.25,A,38.933056,-120.404444,USFS,CA,61,06061,Placer County
2,3,3,FS-1418835,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,...,2024,0.10,A,38.984167,-120.735556,STATE OR PRIVATE,CA,17,06017,El Dorado County
3,4,4,FS-1418845,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,...,1400,0.10,A,38.559167,-119.913333,USFS,CA,3,06003,Alpine County
4,5,5,FS-1418847,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,...,1200,0.10,A,38.559167,-119.933056,USFS,CA,3,06003,Alpine County
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251876,2303543,400732952,ICS209_2019_10757785,INTERAGCY,IA-ICS209,ST/C&L,USCAVNC,Ventura County Fire Department,CAVNC,Ventura County Fire Department,...,None,9999.00,G,34.337222,-119.053333,MISSING/NOT SPECIFIED,CA,Ventura,06111,Ventura County
251877,2303544,400732953,ICS209_2019_10762771,INTERAGCY,IA-ICS209,ST/C&L,USCASLU,San Luis Obispo Unit,CASLU,San Luis Obispo Unit,...,None,835.00,E,35.307500,-119.964444,MISSING/NOT SPECIFIED,CA,San Luis Obispo,06079,San Luis Obispo County
251878,2303552,400732962,ICS209_2019_10781965,INTERAGCY,IA-ICS209,ST/C&L,USCASCU,Santa Clara Unit,CASCU,Santa Clara Unit,...,None,2422.00,F,37.472222,-121.249444,MISSING/NOT SPECIFIED,CA,Stanislaus,06099,Stanislaus County
251879,2303557,400732973,ICS209_2019_10802166,INTERAGCY,IA-ICS209,FS,USCAPNF,Plumas National Forest,CAPNF,Plumas National Forest,...,None,54608.00,G,40.053250,-120.668900,MISSING/NOT SPECIFIED,CA,Plumas,06063,Plumas County


In [23]:
fire = pd.read_sql('''
                    select 
                    FIRE_YEAR,
                    DISCOVERY_DATE,
                    DISCOVERY_TIME,
                    NWCG_CAUSE_CLASSIFICATION,
                    NWCG_GENERAL_CAUSE,
                    FIRE_SIZE,
                    FIRE_SIZE_CLASS,
                    LATITUDE,
                    LONGITUDE,
                    FIPS_CODE,
                    FIPS_NAME
                    from fires where STATE = :state
                    ''',conn,params={'state':'CA'})
# fire = pd.read_sql('''
#                     select FIRE_YEAR,DISCOVERY_DATE,DISCOVERY_DOY,
#                         NWCG_CAUSE_CLASSIFICATION,NWCG_GENERAL_CAUSE,
#                         FIRE_SIZE,FIRE_SIZE_CLASS,
#                         LATITUDE,LONGITUDE
#                     from fires
#                     where STATE = :state
#                     ''',conn,params={'state':'CA'})
fire.columns = fire.columns.str.lower()

In [24]:
fire.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251881 entries, 0 to 251880
Data columns (total 11 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   fire_year                  251881 non-null  int64  
 1   discovery_date             251881 non-null  object 
 2   discovery_time             206816 non-null  object 
 3   nwcg_cause_classification  251881 non-null  object 
 4   nwcg_general_cause         251881 non-null  object 
 5   fire_size                  251881 non-null  float64
 6   fire_size_class            251881 non-null  object 
 7   latitude                   251881 non-null  float64
 8   longitude                  251881 non-null  float64
 9   fips_code                  157156 non-null  object 
 10  fips_name                  157156 non-null  object 
dtypes: float64(3), int64(1), object(7)
memory usage: 21.1+ MB


In [25]:
fire.describe().T

,count,mean,std,min,25%,50%,75%,max
fire_year,251881.0,2006.010457,8.302032,1992.000000,1999.000000,2006.000000,2013.000000,2020.0000
fire_size,251881.0,83.082573,3039.451969,0.001000,0.100000,0.200000,1.000000,589368.0000
latitude,251881.0,37.251235,2.559057,32.537406,34.661935,37.396107,39.226593,42.4175
longitude,251881.0,-120.118132,2.130694,-124.402883,-121.680684,-120.491111,-118.345556,-114.1258


In [26]:
fire

,fire_year,discovery_date,discovery_time,nwcg_cause_classification,nwcg_general_cause,fire_size,fire_size_class,latitude,longitude,fips_code,fips_name
0,2005,2/2/2005,1300,Human,Power generation/transmission/distribution,0.10,A,40.036944,-121.005833,06063,Plumas County
1,2004,5/12/2004,0845,Natural,Natural,0.25,A,38.933056,-120.404444,06061,Placer County
2,2004,5/31/2004,1921,Human,Debris and open burning,0.10,A,38.984167,-120.735556,06017,El Dorado County
3,2004,6/28/2004,1600,Natural,Natural,0.10,A,38.559167,-119.913333,06003,Alpine County
4,2004,6/28/2004,1600,Natural,Natural,0.10,A,38.559167,-119.933056,06003,Alpine County
...,...,...,...,...,...,...,...,...,...,...,...
251876,2019,10/31/2019,2115,Missing data/not specified/undetermined,Missing data/not specified/undetermined,9999.00,G,34.337222,-119.053333,06111,Ventura County
251877,2019,5/29/2019,1900,Missing data/not specified/undetermined,Missing data/not specified/undetermined,835.00,E,35.307500,-119.964444,06079,San Luis Obispo County
251878,2019,6/25/2019,2230,Missing data/not specified/undetermined,Missing data/not specified/undetermined,2422.00,F,37.472222,-121.249444,06099,Stanislaus County
251879,2019,9/4/2019,1446,Missing data/not specified/undetermined,Missing data/not specified/undetermined,54608.00,G,40.053250,-120.668900,06063,Plumas County


In [27]:
# pt['lat'] = round(pt.latitude,1)
# pt['lon'] = round(pt.longitude,1)
# fire['lat'] = round(fire.latitude,1)
# fire['lon'] = round(fire.longitude,1)
fire = fire[fire['fips_code'].notna()].copy()
fire['county_code'] = fire.fips_code.str[2:].astype(int)
fire

,fire_year,discovery_date,discovery_time,nwcg_cause_classification,nwcg_general_cause,fire_size,fire_size_class,latitude,longitude,fips_code,fips_name,county_code
0,2005,2/2/2005,1300,Human,Power generation/transmission/distribution,0.10,A,40.036944,-121.005833,06063,Plumas County,63
1,2004,5/12/2004,0845,Natural,Natural,0.25,A,38.933056,-120.404444,06061,Placer County,61
2,2004,5/31/2004,1921,Human,Debris and open burning,0.10,A,38.984167,-120.735556,06017,El Dorado County,17
3,2004,6/28/2004,1600,Natural,Natural,0.10,A,38.559167,-119.913333,06003,Alpine County,3
4,2004,6/28/2004,1600,Natural,Natural,0.10,A,38.559167,-119.933056,06003,Alpine County,3
...,...,...,...,...,...,...,...,...,...,...,...,...
251876,2019,10/31/2019,2115,Missing data/not specified/undetermined,Missing data/not specified/undetermined,9999.00,G,34.337222,-119.053333,06111,Ventura County,111
251877,2019,5/29/2019,1900,Missing data/not specified/undetermined,Missing data/not specified/undetermined,835.00,E,35.307500,-119.964444,06079,San Luis Obispo County,79
251878,2019,6/25/2019,2230,Missing data/not specified/undetermined,Missing data/not specified/undetermined,2422.00,F,37.472222,-121.249444,06099,Stanislaus County,99
251879,2019,9/4/2019,1446,Missing data/not specified/undetermined,Missing data/not specified/undetermined,54608.00,G,40.053250,-120.668900,06063,Plumas County,63


## Combo testing

In [28]:
fire.shape,pt.shape

((157156, 12), (299154, 13))

In [29]:
# fpt = pd.merge(left=fire,right=pt,how='left',left_on=['fire_year','county_code'],right_on=['measurement_year','tree_county_code'])
# fpt

In [30]:
# small_fire = fire[fire.fire_year==2018]
# small_fire

In [31]:
# small_pt = pt[pt.measurement_year==2018]
# small_pt = small_pt.dropna()
# small_pt

In [32]:
# small_fpt = pd.merge(left=small_fire,right=small_pt,how='right',left_on='county_code',right_on='tree_county_code')
# small_fpt

In [33]:
# small_aq = aq[(aq.year==2018)]
# small_aq

In [34]:
# small_aq['sample_duration'] = pd.to_numeric(small_aq['sample_duration'].str.extract('(\d+)', expand=False), errors='coerce').astype(pd.Int64Dtype())
# small_aq

In [35]:
# small_aq.info()

In [36]:
# piv_saq = small_aq.pivot_table(index=['date_local','county_code'],columns='parameter_name',
#                 values=['arithmetic_mean']).reset_index()
# piv_saq

In [37]:
# small_fpt.discovery_date = small_fpt.discovery_date.astype('datetime64[ns]')
# small_fpt

In [38]:
# piv_saq.columns

In [39]:
# small_ca = pd.merge(left=small_fpt,right=piv_saq,how='left',left_on=['discovery_date','county_code'],right_on=[('date_local',''),('county_code','')])
# small_ca

In [40]:
# small_ca.columns.to_list()

In [41]:
# small_ca.columns = [
#     'fire_year',
#     'date',
#     'time',
#     'cause_class',
#     'cause',
#     'fire_size',
#     'fire_size_class',
#     'lat',
#     'long',
#     'fips_code',
#     'county',
#     'county_code1',
#     'measurement_year',
#     'tree_county_code',
#     'latitude_y',
#     'longitude_y',
#     'elevation',
#     'trees_per_acre',
#     'water_on_plot',
#     'species',
#     'species_group',
#     'height',
#     'diameter',
#     'tree_alive',
#     'invasive_sampling',
#     'date_local',
#     'county_code2',
#     'bp_mean',
#     'co_mean',
#     'dp_mean',
#     'temp_mean',
#     'humidity_mean',
#     'wind_direction_mean',
#     'wind_speed_mean']


In [42]:
# small_ca = small_ca[[
#     'date',
#     'time',
#     'cause_class',
#     'cause',
#     'fire_size',
#     'fire_size_class',
#     'lat',
#     'long',
#     'elevation',
#     'county',
#     'trees_per_acre',
#     'water_on_plot',
#     'species',
#     'species_group',
#     'height',
#     'diameter',
#     'tree_alive',
#     'invasive_sampling',
#     'co_mean',
#     'temp_mean',
#     'humidity_mean',
#     'wind_direction_mean',
#     'wind_speed_mean']]
# small_ca.head()

In [43]:
# # Drop rows with missing data across all columns
# small_ca = small_ca.dropna()
# small_ca

In [44]:
# # map easy categorical values to 1 and 0
# small_ca.tree_alive = small_ca.tree_alive.map({'Live tree':1,'Dead tree':0})
# small_ca.invasive_sampling = small_ca.invasive_sampling.map({'Invasive plant data collected on all accessible land conditions':1,'Not collecting invasive plant data':0})
# small_ca

In [45]:
# small_ca.to_csv('ca_fire2018.csv',index=False)

In [46]:
spt = pt[pt.measurement_year==2018]
spt = spt.dropna()
spt

,measurement_year,tree_county_code,latitude,longitude,elevation,trees_per_acre_unadjusted,water_on_plot_code_name,species_common_name,species_group_code_name,total_height,current_diameter,tree_status_code_name,invasive_sampling_status_code_name
23,2018,51,37.691216,-118.310455,10500,0.999188,None - no water sources within the accessible ...,limber pine,Other western softwoods,39,32.400002,Live tree,Invasive plant data collected on all accessibl...
32,2018,107,36.365822,-118.566055,10700,0.999188,None - no water sources within the accessible ...,foxtail pine,Other western softwoods,35,31.100000,Live tree,Invasive plant data collected on all accessibl...
68,2018,107,36.365822,-118.566055,10700,0.999188,None - no water sources within the accessible ...,foxtail pine,Other western softwoods,57,46.799999,Live tree,Invasive plant data collected on all accessibl...
73,2018,51,37.691216,-118.310455,10500,0.999188,None - no water sources within the accessible ...,limber pine,Other western softwoods,65,45.400002,Live tree,Invasive plant data collected on all accessibl...
96,2018,107,36.365822,-118.566055,10700,0.999188,None - no water sources within the accessible ...,foxtail pine,Other western softwoods,39,32.500000,Live tree,Invasive plant data collected on all accessibl...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
299091,2018,107,36.254704,-118.594475,8700,6.018046,None - no water sources within the accessible ...,western white pine,Western white pine,45,10.300000,Dead tree,Invasive plant data collected on all accessibl...
299099,2018,43,37.623566,-119.546165,8700,0.999188,None - no water sources within the accessible ...,western white pine,Western white pine,83,37.200001,Live tree,Not collecting invasive plant data
299101,2018,109,37.870834,-119.609947,8700,0.999188,None - no water sources within the accessible ...,western white pine,Western white pine,97,24.400000,Live tree,Not collecting invasive plant data
299103,2018,43,37.623566,-119.546165,8700,0.999188,None - no water sources within the accessible ...,western white pine,Western white pine,99,33.400002,Live tree,Not collecting invasive plant data


In [47]:
# piv_spt = spt.pivot_table(index=['measurement_year','tree_county_code','latitude','longitude'],
#         columns=['species_group_code_name','water_on_plot_code_name'],
#         values=['trees_per_acre_unadjusted','total_height','current_diameter','elevation']).reset_index()
# piv_spt

## Fire and Tree Combo

In [48]:
fpt = None
for year in range(2001,2019):
    # fire
    sm_fire = fire[fire.fire_year==year]
    # tree
    sm_pt = pt[pt.measurement_year==year]
    sm_pt = sm_pt.dropna()
    # fire tree combine
    sm_fpt = pd.merge(left=sm_fire,right=sm_pt,how='right',left_on='county_code',right_on='tree_county_code')
    fpt = sm_fpt if fpt is None else pd.concat([fpt,sm_fpt],ignore_index=True)
fpt.discovery_date = fpt.discovery_date.astype('datetime64[ns]')
fpt

,fire_year,discovery_date,discovery_time,nwcg_cause_classification,nwcg_general_cause,fire_size,fire_size_class,latitude_x,longitude_x,fips_code,...,longitude_y,elevation,trees_per_acre_unadjusted,water_on_plot_code_name,species_common_name,species_group_code_name,total_height,current_diameter,tree_status_code_name,invasive_sampling_status_code_name
0,2001.0,2001-02-02,1315,Human,Debris and open burning,0.10,A,37.485500,-118.332900,06051,...,-118.257523,9900,6.018046,None - no water sources within the accessible ...,limber pine,Other western softwoods,37,12.6,Live tree,Invasive plant data collected on all accessibl...
1,2001.0,2001-05-24,1359,Natural,Natural,0.30,B,38.123000,-118.917100,06051,...,-118.257523,9900,6.018046,None - no water sources within the accessible ...,limber pine,Other western softwoods,37,12.6,Live tree,Invasive plant data collected on all accessibl...
2,2001.0,2001-07-02,1500,Natural,Natural,75.00,C,37.902200,-118.876200,06051,...,-118.257523,9900,6.018046,None - no water sources within the accessible ...,limber pine,Other western softwoods,37,12.6,Live tree,Invasive plant data collected on all accessibl...
3,2001.0,2001-07-02,1650,Natural,Natural,5.00,B,38.118500,-119.052900,06051,...,-118.257523,9900,6.018046,None - no water sources within the accessible ...,limber pine,Other western softwoods,37,12.6,Live tree,Invasive plant data collected on all accessibl...
4,2001.0,2001-07-02,1715,Natural,Natural,0.10,A,38.311600,-119.011000,06051,...,-118.257523,9900,6.018046,None - no water sources within the accessible ...,limber pine,Other western softwoods,37,12.6,Live tree,Invasive plant data collected on all accessibl...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43825605,2018.0,2018-06-27,0916,Missing data/not specified/undetermined,Missing data/not specified/undetermined,0.82,B,37.473869,-119.843090,06043,...,-119.546165,8700,0.999188,None - no water sources within the accessible ...,western white pine,Western white pine,94,29.4,Live tree,Not collecting invasive plant data
43825606,2018.0,2018-09-01,2242,Missing data/not specified/undetermined,Missing data/not specified/undetermined,0.10,A,37.460468,-119.878765,06043,...,-119.546165,8700,0.999188,None - no water sources within the accessible ...,western white pine,Western white pine,94,29.4,Live tree,Not collecting invasive plant data
43825607,2018.0,2018-11-18,1327,Missing data/not specified/undetermined,Missing data/not specified/undetermined,1.00,B,37.585661,-120.287601,06043,...,-119.546165,8700,0.999188,None - no water sources within the accessible ...,western white pine,Western white pine,94,29.4,Live tree,Not collecting invasive plant data
43825608,2018.0,2018-06-12,0302,Missing data/not specified/undetermined,Missing data/not specified/undetermined,0.02,A,37.478067,-119.976529,06043,...,-119.546165,8700,0.999188,None - no water sources within the accessible ...,western white pine,Western white pine,94,29.4,Live tree,Not collecting invasive plant data


## Air Pivot

In [49]:
caq = None
for year in range(2001,2019):
    # air quality
    sm_aq = aq[(aq.year==year)]
    # pivot so row is each county on date has mean of parameters
    piv_aq = sm_aq.pivot_table(index=['date_local','county_code'],columns='parameter_name',
                    values=['arithmetic_mean']).reset_index()
    caq = piv_aq if caq is None else pd.concat([caq,piv_aq],ignore_index=True)
caq

date_local county_code     arithmetic_mean                  \
parameter_name                        Barometric pressure Carbon monoxide   
0              2001-01-01           1                 NaN        1.249493   
1              2001-01-01           5                 NaN        0.612198   
2              2001-01-01           7         1011.120833        1.887802   
3              2001-01-01           9                 NaN        0.274034   
4              2001-01-01          11         1019.216667             NaN   
...                   ...         ...                 ...             ...   
313036         2018-12-31         101                 NaN             NaN   
313037         2018-12-31         107         1002.147917             NaN   
313038         2018-12-31         109                 NaN             NaN   
313039         2018-12-31         111          979.208333             NaN   
313040         2018-12-31         113                 NaN             NaN   

                                                                 \
parameter_name Dew Point Outdoor Temperature Relative Humidity    
0                    NaN                 NaN                NaN   
1                    NaN           41.708333                NaN   
2                    NaN           46.402778          77.312500   
3                    NaN           43.458333                NaN   
4                    NaN           44.270833                NaN   
...                  ...                 ...                ...   
313036               NaN           44.420833          45.100000   
313037               NaN           41.589583          77.058333   
313038               NaN           40.920833          53.462500   
313039               NaN           50.256944          46.937500   
313040               NaN           46.820833          49.400000   

                                                                  
parameter_name Wind Direction - Resultant Wind Speed - Resultant  
0                                     NaN                    NaN  
1                              155.958333               1.583333  
2                              176.625000               1.333333  
3                              160.583333               1.958333  
4                              225.625000               1.875000  
...                                   ...                    ...  
313036                         320.000000               6.225000  
313037                         140.275000               2.414583  
313038                         186.666667               1.662500  
313039                          91.923611               6.345833  
313040                         335.125000              14.983333  

[313041 rows x 9 columns]

## Fire Tree Air Combo

In [50]:
# air quality fire tree combine
ca = pd.merge(left=fpt,right=caq,how='left',left_on=['discovery_date','county_code'],right_on=[('date_local',''),('county_code','')])

/var/folders/ml/8_q6055n29vf0c36kmj6w1900000gn/T/ipykernel_6000/3806272112.py:2: FutureWarning: merging between different levels is deprecated and will be removed in a future version. (1 levels on the left, 2 on the right)
  ca = pd.merge(left=fpt,right=caq,how='left',left_on=['discovery_date','county_code'],right_on=[('date_local',''),('county_code','')])


In [51]:
ca.head()

,fire_year,discovery_date,discovery_time,nwcg_cause_classification,nwcg_general_cause,fire_size,fire_size_class,latitude_x,longitude_x,fips_code,...,invasive_sampling_status_code_name,"(date_local, )","(county_code, )","(arithmetic_mean, Barometric pressure)","(arithmetic_mean, Carbon monoxide)","(arithmetic_mean, Dew Point)","(arithmetic_mean, Outdoor Temperature)","(arithmetic_mean, Relative Humidity )","(arithmetic_mean, Wind Direction - Resultant)","(arithmetic_mean, Wind Speed - Resultant)"
0,2001.0,2001-02-02,1315,Human,Debris and open burning,0.1,A,37.4855,-118.3329,06051,...,Invasive plant data collected on all accessibl...,2001-02-02,51.0,1027.375000,0.587500,NaN,33.783333,NaN,NaN,NaN
1,2001.0,2001-05-24,1359,Natural,Natural,0.3,B,38.1230,-118.9171,06051,...,Invasive plant data collected on all accessibl...,2001-05-24,51.0,1023.833333,0.272282,NaN,65.858333,2.916667,NaN,NaN
2,2001.0,2001-07-02,1500,Natural,Natural,75.0,C,37.9022,-118.8762,06051,...,Invasive plant data collected on all accessibl...,2001-07-02,51.0,1025.125000,NaN,NaN,72.908333,1.708333,NaN,NaN
3,2001.0,2001-07-02,1650,Natural,Natural,5.0,B,38.1185,-119.0529,06051,...,Invasive plant data collected on all accessibl...,2001-07-02,51.0,1025.125000,NaN,NaN,72.908333,1.708333,NaN,NaN
4,2001.0,2001-07-02,1715,Natural,Natural,0.1,A,38.3116,-119.0110,06051,...,Invasive plant data collected on all accessibl...,2001-07-02,51.0,1025.125000,NaN,NaN,72.908333,1.708333,NaN,NaN


In [52]:
ca.columns

Index([                                      'fire_year',
                                        'discovery_date',
                                        'discovery_time',
                             'nwcg_cause_classification',
                                    'nwcg_general_cause',
                                             'fire_size',
                                       'fire_size_class',
                                            'latitude_x',
                                           'longitude_x',
                                             'fips_code',
                                             'fips_name',
                                           'county_code',
                                      'measurement_year',
                                      'tree_county_code',
                                            'latitude_y',
                                           'longitude_y',
                                             'elevation',
              

In [53]:
ca.columns = [
    'fire_year',
    'date',
    'time',
    'cause_class',
    'cause',
    'fire_size',
    'fire_size_class',
    'lat',
    'long',
    'fips_code',
    'county',
    'county_code1',
    'measurement_year',
    'tree_county_code',
    'latitude_y',
    'longitude_y',
    'elevation',
    'trees_per_acre',
    'water_on_plot',
    'species',
    'species_group',
    'height',
    'diameter',
    'tree_alive',
    'invasive_sampling',
    'date_local',
    'county_code2',
    'bp_mean',
    'co_mean',
    'dp_mean',
    'temp_mean',
    'humidity_mean',
    'wind_direction_mean',
    'wind_speed_mean']
ca = ca[[
    'date',
    'time',
    'cause_class',
    'cause',
    'fire_size',
    'fire_size_class',
    'lat',
    'long',
    'elevation',
    'county',
    'trees_per_acre',
    'water_on_plot',
    'species',
    'species_group',
    'height',
    'diameter',
    'tree_alive',
    'invasive_sampling',
    'co_mean',
    'temp_mean',
    'humidity_mean',
    'wind_direction_mean',
    'wind_speed_mean']]
ca.shape

(43825610, 23)

In [54]:
ca.isnull().sum()

date                        574
time                      73682
cause_class                 574
cause                       574
fire_size                   574
fire_size_class             574
lat                         574
long                        574
elevation                     0
county                      574
trees_per_acre                0
water_on_plot                 0
species                       0
species_group                 0
height                        0
diameter                      0
tree_alive                    0
invasive_sampling             0
co_mean                27614377
temp_mean              18502237
humidity_mean          22246284
wind_direction_mean    20839382
wind_speed_mean        20756342
dtype: int64

In [56]:
# Drop rows with missing data
# ca[(ca.co_mean.notna())&(ca.wind_speed_mean.notna())&(ca.humidity_mean.notna())&(ca.wind_direction_mean.notna())&(ca.time.notna())].isnull().sum()
ca = ca[(ca.co_mean.notna())&(ca.wind_speed_mean.notna())&(ca.humidity_mean.notna())&(ca.wind_direction_mean.notna())&(ca.time.notna())]
ca.shape

(9947589, 23)

In [57]:
ca.date.dt.year.value_counts().sort_index()

2001    507483
2002    643029
2003    542576
2004    676210
2005    865066
2006    697532
2007    905596
2008    477745
2009    626314
2010    246877
2011    385769
2012    242668
2013    230304
2014    131273
2015    314165
2016    635258
2017    931809
2018    887915
Name: date, dtype: int64

In [58]:
# map easy categorical values to 1 and 0
ca.tree_alive = ca.tree_alive.map({'Live tree':1,'Dead tree':0})
ca.invasive_sampling = ca.invasive_sampling.map({'Invasive plant data collected on all accessible land conditions':1,'Not collecting invasive plant data':0})
ca.sample(5)

,date,time,cause_class,cause,fire_size,fire_size_class,lat,long,elevation,county,...,species_group,height,diameter,tree_alive,invasive_sampling,co_mean,temp_mean,humidity_mean,wind_direction_mean,wind_speed_mean
22571155,2009-11-23,1317,Human,Misuse of fire by a minor,3.00,B,33.966576,-117.959850,7900,Los Angeles County,...,True fir,68,26.600000,1,1,0.796991,59.200000,55.652778,176.196474,2.615705
6112560,2004-05-30,2108,Human,Missing data/not specified/undetermined,0.30,B,36.248889,-120.626111,8300,Fresno County,...,True fir,135,31.500000,1,1,0.236413,73.291667,45.322917,281.041667,2.958333
6169081,2004-06-06,1145,Human,Equipment and vehicle use,1.00,B,36.903056,-119.665000,5500,Fresno County,...,Sugar pine,112,39.799999,1,1,0.212704,77.000000,43.388889,320.083333,6.541667
43087462,2018-02-16,1232,Human,Debris and open burning,0.01,A,36.561767,-119.713787,3500,Fresno County,...,Ponderosa and Jeffrey pines,49,13.900000,0,1,0.400243,51.739881,52.743056,173.110119,2.905952
6577399,2004-04-23,1831,Human,Equipment and vehicle use,0.10,A,36.175000,-119.018889,5300,Tulare County,...,Oak,26,6.500000,0,1,0.367845,58.808334,43.000000,172.458333,5.220833


In [59]:
ca.to_csv('ca_fire.csv',index=False)